In [8]:
import duckdb
import pandas as pd

In [ ]:
# Connect to the database
con = duckdb.connect("publications.db")

In [10]:
# Have a look at the data stored in the csv file
con.sql("SELECT id, title, abstract, year, scope, pillar, research_category  FROM 'publications_curated.csv'").show()

┌────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
# This would be a way to create the table and load the data in one step, but it doesn't allow us to set the PublicationID as the primary key
# con.sql("CREATE TABLE publications_raw AS SELECT id, title, abstract, year, scope, pillar, research_category FROM 'publications_data.csv'")

# This line deletes the created table
# con.sql("DROP TABLE IF EXISTS publications_raw")

In [12]:
# In order to have the Publication ID as the primary key, we need to create the table in two steps:
# Step 1: create the table with the schema
con.sql("""
    CREATE TABLE publications_raw (
        id VARCHAR PRIMARY KEY,
        title VARCHAR,
        abstract VARCHAR,
        year INT,
        scope VARCHAR,
        pillar VARCHAR,
        research_category VARCHAR
    )
""")

# Step 2: load the CSV into it
con.sql("INSERT INTO publications_raw SELECT id, title, abstract, year, scope, pillar, research_category FROM 'publications_curated.csv'")

In [13]:
con.sql("DESCRIBE publications_raw").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ title             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ abstract          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ scope             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pillar            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ research_category │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [14]:
# Extract query results that had not been present in the database yet
con.sql("""
    SELECT COUNT(*)
    FROM publications_raw
    WHERE scope = 'in'
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1042 │
└──────────────┘



In [15]:
# Convert to pandas dataframe
df = con.sql("SELECT * FROM publications_raw").df()

In [16]:
# Filter for in scope publications
df[df['scope'] == 'in']

,id,title,abstract,year,scope,pillar,research_category
0,pub.1192791591,Mushroom: an emerging source for next generati...,"Background: In recent years, plant-based and a...",2025,in,PB,Ingredient optimisation
1,pub.1187762379,Plant-Based Alternatives to Meat Products,Animal proteins have been used in the formulat...,2025,in,PB,Other
2,pub.1193391974,Influence of processing on protein quality and...,While meat is an established source of high-qu...,2025,in,PB,Ingredient optimisation
3,pub.1188899046,Solid State Fermentation—A Promising Approach ...,The increasing demand for sustainable dietary ...,2025,in,F,Other
4,pub.1193352421,"Physicochemical, Microbiological and Sensory E...",The bioactive properties of a phenolic extract...,2025,in,PB,End product formulation
...,...,...,...,...,...,...,...
4713,pub.1191431512,Processing‐Driven Changes in Potato Protein‐St...,ABSTRACT This study investigates the digestibi...,2025,in,PB,Ingredient optimisation
4718,pub.1188788048,Effect of pH and salt on the structure and the...,As an increasingly important ingredient in sus...,2025,in,PB,Ingredient optimisation
4726,pub.1184328597,Extraction of Protein-Enriched Fractions from ...,Simultaneous extraction of oil and protein fro...,2025,in,PB,Ingredient optimisation
4728,pub.1181755693,Characterization of rapeseed protein supramole...,Aqueous extraction of rapeseed has the potenti...,2025,in,PB,Ingredient optimisation


In [17]:
con.close()